# 06 · Resource benchmark — §4, §6

Latency and peak memory depend on the machine, but not on trained weights. So every architecture is built fresh and profiled back to back here — no checkpoints needed, and every row of the §6 table is measured on the same hardware, whoever trained which model where. A local machine is recommended: its CPU model is stable and can be named in the report.

| | |
|---|---|
| **Owner** | Member 1 |
| **Reads** | each run's experiment config (architecture only) and `history.json` (epoch times) |
| **Writes (official)** | `results/metrics/<run_id>/resources.json` for all six runs |
| **Run it** | once, after 03–05's official runs are committed — in **one session, on one CPU** |

In [ ]:
# ── Cell 1 · Colab bootstrap ─ does nothing when run locally ───────────────
# Locally the package is already installed (pip install -e .), so this is skipped.
# On Colab it clones the repo, installs it and makes it importable.
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import os
    import subprocess

    BRANCH = "main"  # <- your branch, to run work that is not merged yet
    REPO_URL = "https://github.com/ThejithaR/EN3150-Assignment-03-CNN.git"
    REPO_DIR = "/content/EN3150-Assignment-03-CNN"

    if not os.path.exists(REPO_DIR):
        env = dict(os.environ)
        try:  # only needed if the repo is private; the token never leaves the environment
            from google.colab import userdata

            token = userdata.get("GH_TOKEN")
            if token:
                env["GH_TOKEN"] = token
        except Exception:
            pass
        helper = "!f() { echo username=x-access-token; echo password=$GH_TOKEN; }; f"
        subprocess.run(
            ["git", "-c", f"credential.helper={helper}", "clone", "--quiet",
             "--branch", BRANCH, REPO_URL, REPO_DIR],
            env=env,
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR], check=True)
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")  # an editable install is only seen after a restart
    os.chdir(REPO_DIR)
    print(f"Colab ready: {REPO_DIR} @ {BRANCH}")

In [ ]:
# ── Cell 2 · Setup ─────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

# MODE decides how much runs and whether anything is written:
#   "synthetic"  generated data, 2 epochs, seconds         - before EuroSAT exists
#   "debug"      5% of EuroSAT, 2 epochs                   - find bugs step by step
#   "official"   the full run - the ONLY mode that writes  - then Restart & Run All
MODE = "synthetic"

from edgecnn.config import load_config
from edgecnn.contracts import paths, schema
import dataclasses
import platform

import pandas as pd

from edgecnn.evaluation import profile_all
from edgecnn.utils import describe_device, resolve_device

## Runs to profile

Every run that appears in a report table.

In [ ]:
tables = load_config("configs/experiments/model_b__adam.yaml").section("reporting")["tables"]
RUN_IDS = sorted({run for table in tables for run in table["runs"]})
RUN_IDS

## Hardware

This is the machine the §6 latency column describes.

In [ ]:
print(platform.processor() or platform.machine())
print(describe_device(resolve_device("cpu")))

## Profile every architecture — one session

In [ ]:
profiles = profile_all(RUN_IDS, mode=MODE)

## Results

In [ ]:
pd.DataFrame([dataclasses.asdict(p) for p in profiles.values()]).set_index("run_id")

## Official-run checklist

- [ ] `MODE = "official"` in the setup cell
- [ ] Kernel → **Restart & Run All** finished without errors
- [ ] every output is what you expect — no stray warnings or giant prints
- [ ] 03, 04 and 05 official results were committed before this run
- [ ] ran on CPU, in one session, on one machine — named in the report
- [ ] `pytest tests/contracts` is green
- [ ] commit this notebook **with its outputs**, together with the files it wrote

In [ ]:
# ── Colab only: send this run's results back to GitHub ─────────────────────
# Does nothing locally, or in synthetic / debug mode. Procedure: notebooks/README.md
if IN_COLAB and MODE == "official":
    from edgecnn.utils import push_results

    push_results(
        [paths.metrics_dir(run_id) / "resources.json" for run_id in RUN_IDS],
        "Add resource profiles (Colab)",
    )